# WBTC/WETH Dataset Background Analysis

This notebook reads the curated dataset built by the CLI and reproduces the core tables and charts for the assignment report.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from mev_dataset.config import load_market_config

config = load_market_config('../config/markets.yaml')
curated_dir = Path('../') / config.curated_data_dir
report_dir = Path('../') / config.report_dir
report_dir.mkdir(parents=True, exist_ok=True)
pool_state = pd.read_parquet(curated_dir / 'pool_state_1m.parquet')
arb_labels = pd.read_parquet(curated_dir / 'arb_labels_1m.parquet')
pool_state['timestamp'] = pd.to_datetime(pool_state['timestamp'], utc=True)
arb_labels['timestamp'] = pd.to_datetime(arb_labels['timestamp'], utc=True)
pool_state.head()

In [ ]:
price_pivot = pool_state.pivot(index='timestamp', columns='dex', values='mid_price_eth_per_btc')
ax = price_pivot.plot(figsize=(12, 5), title='WBTC/WETH Pool Price Series')
ax.set_ylabel('ETH per BTC')
plt.tight_layout()
plt.show()

In [ ]:
spread_bps = (price_pivot.iloc[:, 0] / price_pivot.iloc[:, 1] - 1.0) * 10_000
ax = spread_bps.plot(figsize=(12, 5), title='Cross-DEX Spread')
ax.set_ylabel('Basis points')
plt.tight_layout()
plt.show()

In [ ]:
swap_activity = (
    pool_state.assign(hour_utc=pool_state['timestamp'].dt.hour)
    .groupby(['dex', 'hour_utc'], as_index=False)
    .agg(
        avg_swap_count=('swap_count', 'mean'),
        avg_volume_wbtc=('volume_wbtc', 'mean'),
        avg_volume_weth=('volume_weth', 'mean'),
    )
)
swap_activity

In [ ]:
split_descriptives = (
    arb_labels.groupby('split', as_index=False)
    .agg(
        mean_net_edge_bps=('net_edge_bps', 'mean'),
        median_net_edge_bps=('net_edge_bps', 'median'),
        positive_opportunities=('opportunity_flag', 'sum'),
        observations=('timestamp', 'count'),
    )
)
split_descriptives